In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import time
import pandas as pd

In [11]:
def parse_game_list(driver):
    game_list = []
    game_table = driver.find_element(By.CLASS_NAME, 'collection_table')
    table_header = game_table.find_elements(By.CSS_SELECTOR, 'tbody > tr')[0]
    table_headers = [header.text for header in table_header.find_elements(By.TAG_NAME, 'th')]
    # print(table_headers)
    
    game_rows = game_table.find_elements(By.ID, 'row_')
    assert len(game_rows) > 0, "No games found on the page."
    
    for game_row in game_rows:
        driver.execute_script("arguments[0].scrollIntoView();", game_row) # Scroll to the element to make it visible to the driver
        entries = game_row.find_elements(By.TAG_NAME, 'td')
        game_rank = entries[0].text
        # print(game_rank)
        game_link = entries[1].find_element(By.TAG_NAME, 'a').get_attribute('href')
        game_name = entries[2].find_element(By.TAG_NAME, 'a').text
        game_year = entries[2].find_element(By.TAG_NAME, 'span').text
        geek_rating = entries[3].text
        avg_rating = entries[4].text
        num_voters = entries[5].text
        game_list.append({
            'Board Game Rank': game_rank,
            'Game Link': game_link,
            'Title': game_name,
            'Geek Rating': geek_rating,
            'Avg Rating': avg_rating,
            'Num Voters': num_voters,
            'Year': game_year
        })
    return pd.DataFrame(game_list, columns=['Board Game Rank', 'Game Link', 'Title', 'Geek Rating', 'Avg Rating', 'Num Voters', 'Year'])

In [12]:
MAIN_URL = 'https://boardgamegeek.com/browse/boardgame'

driver = webdriver.Chrome()
driver.get(MAIN_URL)
time.sleep(2)

all_games = []
for i in range(3):
    print(f"Scraping page {i+1}...")
    games = parse_game_list(driver)
    print(f"Found {len(games)} games.")
    all_games.append(games)
    next_page = driver.find_element(By.XPATH, '//*[@id="maincontent"]/form/div/div[1]/a[5]')
    next = next_page.get_attribute('href')
    driver.get(next)
    time.sleep(2)

driver.quit()

Scraping page 0...
Found 100 games.
Scraping page 1...
Found 100 games.
Scraping page 2...
Found 100 games.


In [14]:
bgg_df = pd.concat(all_games)
bgg_df.reset_index(drop=True, inplace=True)
bgg_df.to_csv('data/bgg_data.csv', index=False)

In [16]:
bgg_df.head(10)

,Board Game Rank,Game Link,Title,Geek Rating,Avg Rating,Num Voters,Year
0,1,https://boardgamegeek.com/boardgame/224517/bra...,Brass: Birmingham,8.411,8.59,48362,(2018)
1,2,https://boardgamegeek.com/boardgame/161936/pan...,Pandemic Legacy: Season 1,8.374,8.52,54307,(2015)
2,3,https://boardgamegeek.com/boardgame/174430/glo...,Gloomhaven,8.342,8.58,63272,(2017)
3,4,https://boardgamegeek.com/boardgame/342942/ark...,Ark Nova,8.338,8.53,46931,(2021)
4,5,https://boardgamegeek.com/boardgame/233078/twi...,Twilight Imperium: Fourth Edition,8.235,8.59,24692,(2017)
5,6,https://boardgamegeek.com/boardgame/316554/dun...,Dune: Imperium,8.230,8.43,48349,(2020)
6,7,https://boardgamegeek.com/boardgame/167791/ter...,Terraforming Mars,8.207,8.35,101878,(2016)
7,8,https://boardgamegeek.com/boardgame/115746/war...,War of the Ring: Second Edition,8.191,8.55,22119,(2011)
8,9,https://boardgamegeek.com/boardgame/187645/sta...,Star Wars: Rebellion,8.170,8.42,33268,(2016)
9,10,https://boardgamegeek.com/boardgame/291457/glo...,Gloomhaven: Jaws of the Lion,8.150,8.42,35690,(2020)
